In [ ]:
import numpy as np
from pathlib import Path
import time
import sys
import os

# ── Add RandLA-Net to path ────────────────────────────────────────────────
RANDLA_ROOT = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch")
sys.path.insert(0, str(RANDLA_ROOT))
sys.path.insert(0, str(RANDLA_ROOT / "utils"))
os.chdir(RANDLA_ROOT)

import torch
from model import RandLANet
from utils.ply import write_ply

try:
    from torch_points_kernels import knn as knn_fn
except ImportError:
    from torch_points import knn as knn_fn

# ── Parameters ────────────────────────────────────────────────────────────
INPUT_TXT_DIR   = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Test")
CHECKPOINT_PATH = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/checkpoints/randlanet_Purdue.pth")
OUTPUT_DIR      = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/results")
D_IN            = 5
NUM_CLASSES     = 10
NUM_NEIGHBORS   = 16
DECIMATION      = 4
NUM_LAYERS      = 5
NUM_POINTS      = 45056    # must be divisible by DECIMATION^NUM_LAYERS = 4^5 = 1024
SEP             = "\t"   
X_COL, Y_COL, Z_COL = 0, 1, 2
FEAT_COLS       = [3, 4]    # Intensity, PointSourceID, GPSTime → D_IN=6
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Helper: load txt ──────────────────────────────────────────────────────
def load_txt(path):
    import pandas as pd
    df   = pd.read_csv(path, sep="\t", header=None, engine="python")
    xyz  = df[[0, 1, 2]].values.astype(np.float32)
    feat = df[[3, 4]].values.astype(np.float32)   # only 2 cols now

    feat_min = feat.min(axis=0, keepdims=True)
    feat_max = feat.max(axis=0, keepdims=True)
    feat     = (feat - feat_min) / (feat_max - feat_min + 1e-8)

    return np.hstack([xyz, feat])   # (N, 5)

# ── Helper: sample / pad ──────────────────────────────────────────────────
def sample_points(pts, n):
    N = len(pts)
    if N >= n:
        idx = np.random.choice(N, n, replace=False)
    else:
        idx = np.concatenate([np.arange(N),
                               np.random.choice(N, n - N, replace=True)])
    return pts[idx], idx

# ── Helper: precompute inputs dict ────────────────────────────────────────
def build_inputs(pts, num_layers, num_neighbors, decimation, device):
    """
    Precompute coords, neighbor_indices, sub_idx, interp_idx
    for all encoder layers — mimics data.py transform().
    pts: np.ndarray (N, D_IN)
    """
    import pandas as pd

    coords_list   = []
    neighbor_list = []
    sub_idx_list  = []
    interp_list   = []

    pc = pts[:, :3].copy()   # (N, 3)

    for i in range(num_layers):
        pc_tensor = torch.from_numpy(pc).unsqueeze(0)   # (1, N, 3)
        N_i = pc.shape[0]

        # KNN on current scale
        neighbor_idx, _ = knn_fn(
            pc_tensor.contiguous(),
            pc_tensor.contiguous(),
            num_neighbors
        )   # (1, N, K)

        # Subsampled points
        N_sub   = N_i // decimation
        pool_i  = neighbor_idx[:, :N_sub, :]            # (1, N_sub, K)
        pc_sub  = pc[:N_sub, :]

        # Upsample index: for each point in pc, find nearest in pc_sub
        pc_sub_tensor = torch.from_numpy(pc_sub).unsqueeze(0)
        up_i, _  = knn_fn(
            pc_sub_tensor.contiguous(),
            pc_tensor.contiguous(),
            1
        )   # (1, N, 1)

        coords_list.append(pc_tensor)
        neighbor_list.append(neighbor_idx.long())
        sub_idx_list.append(pool_i.long())
        interp_list.append(up_i.long())

        pc = pc_sub   # next layer uses subsampled points

    inputs = {
        'features':         torch.from_numpy(pts).unsqueeze(0),   # (1, N, D_IN)
        'coords':           coords_list,
        'neighbor_indices': neighbor_list,
        'sub_idx':          sub_idx_list,
        'interp_idx':       interp_list,
    }
    return inputs

# ── Load model ────────────────────────────────────────────────────────────
t0 = time.time()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")

print("Loading model...")
model = RandLANet(D_IN, NUM_CLASSES, NUM_NEIGHBORS, DECIMATION, device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"  Checkpoint : '{CHECKPOINT_PATH.name}'  epoch={checkpoint['epoch']}")

# ── Process each tile ─────────────────────────────────────────────────────
tile_files = sorted(INPUT_TXT_DIR.glob("*_local_selected.txt"))
print(f"\nFound {len(tile_files)} tile(s)")

for tile_path in tile_files:
    print(f"\nProcessing '{tile_path.name}' ...")

    pts = load_txt(tile_path)
    print(f"  Points: {len(pts):,}  |  Features: {pts.shape[1]}")

    pts_sampled, _ = sample_points(pts, NUM_POINTS)
    print(f"  Sampled to: {NUM_POINTS:,}")

    print("  Building inputs (KNN) ...")
    inputs = build_inputs(pts_sampled, NUM_LAYERS, NUM_NEIGHBORS, DECIMATION, device)

    print("  Running inference ...")
    with torch.no_grad():
        scores      = model(inputs)                              # (1, N, NUM_CLASSES)
        predictions = torch.max(scores, dim=-1).indices          # (1, N)
        predictions = predictions.squeeze(0).cpu().numpy()       # (N,)

    print(f"  Unique classes predicted: {np.unique(predictions).tolist()}")

    # ── Save PLY ──────────────────────────────────────────────────────
    xyz_out  = pts_sampled[:, :3]
    ply_path = OUTPUT_DIR / (tile_path.stem + "_pred.ply")
    write_ply(str(ply_path),
              [xyz_out, predictions.astype(np.int32)],
              ["x", "y", "z", "class"])
    print(f"  PLY → '{ply_path.name}'")

    # ── Save TXT ──────────────────────────────────────────────────────
    txt_path = OUTPUT_DIR / (tile_path.stem + "_pred.txt")
    np.savetxt(str(txt_path),
               np.hstack([xyz_out, predictions.reshape(-1, 1)]),
               fmt="%.4f %.4f %.4f %d")
    print(f"  TXT → '{txt_path.name}'")

t1 = time.time()
print(f"\nAll done. Time elapsed: {t1 - t0:.1f}s")

Using device : cuda:0
Loading model...
  Checkpoint : 'randlanet_Purdue.pth'  epoch=700

Found 1 tile(s)

Processing 'tile_0010_local_selected.txt' ...
  Points: 3,092,589  |  Features: 5
  Sampled to: 45,056
  Building inputs (KNN) ...
  Running inference ...
  Unique classes predicted: [5, 6, 8, 9]
  PLY → 'tile_0010_local_selected_pred.ply'
  TXT → 'tile_0010_local_selected_pred.txt'

All done. Time elapsed: 16.9s
